# Analysis — `n1m5_T15_obs5000_seed42`

Compare the **credulous** vs **vigilant** listener under two speakers (informative, persuasive), across all 9 true thetas. The figure is **2 rows × 9 columns**:
- Row 1: speaker is `inf` (informative).
- Row 2: speaker is `persp` (persuasive, `pers+`).
- Column k: true θ = 0.1·k.
- x = round (0..15), y = the **median across trajectories of the per-trajectory posterior median(θ)**. Two lines + 95% CI shading: blue = credulous, orange = vigilant. Round 0 is the (flat) prior; recorded round t=0 is plotted as round 1.

The notebook is factorized: **Load → Compute → Aggregate → Visualize**.


## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
from pathlib import Path

# Bootstrap repo root onto sys.path so absolute imports work from a notebook.
HERE = Path.cwd().resolve()
# analyze.ipynb -> n1m5.../ -> simulation_experiments/ -> simulations/ -> models/ -> repo root
REPO_ROOT = HERE.parents[3]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

from models.simulations.simulation_experiments.n1m5_T15_obs5000_seed42.io import load_beliefs

DATA_ROOT = HERE / 'raw_do_not_track'
print('DATA_ROOT =', DATA_ROOT)
assert DATA_ROOT.is_dir(), f'expected experiment data at {DATA_ROOT}'

In [ ]:
# Identifiers — must match the directory names produced by run.py.
SPEAKERS = {
    'inf':   'inf_L1strat_a3_b1_uiF',
    'persp': 'persp_L1strat_a3_b0_uiF',
}
LISTENERS = {
    'credulous': 'credulous_L1coop_a3_uiF',
    'vigilant':  'vigilant_L1strat_a3_uiF',
}

# TRUE_THETAS is the set of true thetas at which observations were sampled
# (the figure columns). The agents' belief grid is wider — it also includes
# 0.0 and 1.0 — and shows up in the loaded data as belief_ds.theta
# (length 11). Don't confuse the two.
TRUE_THETAS = [round(0.1 * k, 1) for k in range(1, 10)]   # [0.1, ..., 0.9]

# Plot config
LISTENER_COLORS = {'credulous': 'tab:blue', 'vigilant': 'tab:orange'}
SPEAKER_TITLES = {'inf': 'Informative speaker', 'persp': 'Persuasive (pers+) speaker'}


## 1. Load — read belief Datasets for the (speaker × listener) cells we care about

In [ ]:
def load_belief_grid(data_root, speakers, listeners):
    """Return a dict {(spk_key, lst_key): xr.Dataset} for every (speaker, listener) pair."""
    out = {}
    for spk_key, spk_dir in speakers.items():
        for lst_key, lst_dir in listeners.items():
            path = data_root / spk_dir / lst_dir
            out[(spk_key, lst_key)] = load_beliefs(path)
    return out

beliefs = load_belief_grid(DATA_ROOT, SPEAKERS, LISTENERS)
for k, ds in beliefs.items():
    print(f'{k}: sizes={dict(ds.sizes)} path={ds.attrs["execution_path"]}')

## 2. Compute — posterior median per trajectory, with the flat prior prepended at round 0

Per (theta_true, traj, round) we compute the **posterior median** of θ: the value $\theta^\star$ at which the listener's posterior CDF crosses 0.5, with linear interpolation between adjacent grid points. Round 0 is prepended as the prior median (= median of the θ grid = 0.5 for the uniform prior over $\{0.0, 0.1, \dots, 1.0\}$).

We collapse `obs_idx` × `utt_idx` into a single trajectory axis so downstream aggregation is straightforward; with `n_utt_seq=1` for this experiment the two axes carry the same information.


In [ ]:
def _posterior_median_along_last(probs, grid):
    """
    Median of a discrete distribution via linear CDF interpolation.
    probs: shape (..., n_grid), normalized along last axis.
    grid:  1-D ascending array of length n_grid.
    Returns: shape (...,) median value.
    """
    cdf = np.cumsum(probs, axis=-1)
    idx = (cdf >= 0.5).argmax(axis=-1)                      # first crossing
    idx_lo = np.clip(idx - 1, 0, None)
    cdf_hi = np.take_along_axis(cdf, idx[..., None], axis=-1).squeeze(-1)
    cdf_lo = np.where(idx == 0, 0.0,
                      np.take_along_axis(cdf, idx_lo[..., None], axis=-1).squeeze(-1))
    p = cdf_hi - cdf_lo
    frac = np.where(p > 1e-15, (0.5 - cdf_lo) / np.maximum(p, 1e-15), 0.0)
    frac = np.clip(frac, 0.0, 1.0)
    theta_hi = grid[idx]
    theta_lo = np.where(idx == 0, grid[0], grid[idx_lo])
    return theta_lo + frac * (theta_hi - theta_lo)


def posterior_median_with_prior(belief_ds):
    """Posterior median(θ) per trajectory + synthetic round-0 prior median."""
    grid = belief_ds.theta.values
    bt = belief_ds.belief_theta.stack(traj=('obs_idx', 'utt_idx')).transpose(
        'theta_true', 'traj', 't', 'theta'
    )
    arr = bt.values                                         # (theta_true, traj, t, theta)
    med = _posterior_median_along_last(arr, grid)           # (theta_true, traj, t)

    prior_med = float(_posterior_median_along_last(
        np.full(grid.shape, 1.0 / len(grid)), grid))
    n_thetas, n_traj, _ = med.shape
    prior_block = np.full((n_thetas, n_traj, 1), prior_med, dtype=med.dtype)
    return np.concatenate([prior_block, med], axis=2)


post_med = {k: posterior_median_with_prior(ds) for k, ds in beliefs.items()}

for k, arr in post_med.items():
    print(f'{k}: shape={arr.shape} (theta_true, traj, round)  '
          f'range=[{arr.min():.3f}, {arr.max():.3f}]')


## 3. Aggregate — median across trajectories + 95% interval

`summarize_traj` returns the cross-trajectory median of the per-trajectory posterior median, plus the 2.5 / 97.5 percentile band.


In [ ]:
def summarize_traj(arr, lo_pct=2.5, hi_pct=97.5):
    """arr: (theta_true, traj, round). Returns (median, lo, hi), each (theta_true, round)."""
    median = np.median(arr, axis=1)
    lo = np.percentile(arr, lo_pct, axis=1)
    hi = np.percentile(arr, hi_pct, axis=1)
    return median, lo, hi


summaries_post_med = {k: summarize_traj(arr) for k, arr in post_med.items()}

# Sanity check at theta_true=0.5 (idx 4), final round
print('Persuasive speaker, theta_true=0.5, last round (median of posterior medians):')
for lst in ['credulous', 'vigilant']:
    med, lo, hi = summaries_post_med[('persp', lst)]
    print(f'  {lst:10s} central={med[4, -1]:.3f}  95%CI=[{lo[4, -1]:.3f}, {hi[4, -1]:.3f}]')


## 4. Visualize

In [ ]:
def plot_belief_grid(summaries, thetas, speakers, listeners,
                     listener_colors, speaker_titles,
                     y_label, fig_title=None):
    n_rows = len(speakers)
    n_cols = len(thetas)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(2.0 * n_cols, 2.5 * n_rows),
        sharex=True, sharey=True,
        squeeze=False,
    )

    any_central = next(iter(summaries.values()))[0]
    n_rounds = any_central.shape[1]
    rounds = np.arange(n_rounds)

    for row, spk_key in enumerate(speakers):
        for col, theta_true in enumerate(thetas):
            ax = axes[row, col]
            ax.axhline(theta_true, color='black', linestyle='--',
                       linewidth=0.8, alpha=0.6)

            for lst_key in listeners:
                central, lo, hi = summaries[(spk_key, lst_key)]
                color = listener_colors[lst_key]
                ax.plot(rounds, central[col], color=color, label=lst_key, linewidth=1.6)
                ax.fill_between(rounds, lo[col], hi[col], color=color, alpha=0.18)

            if row == 0:
                ax.set_title(f'$\\theta_{{true}}={theta_true}$', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'{speaker_titles[spk_key]}\n{y_label}', fontsize=9)
            if row == n_rows - 1:
                ax.set_xlabel('round')
            ax.set_ylim(0, 1)
            ax.set_xlim(0, n_rounds - 1)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if fig_title is not None:
        fig.suptitle(fig_title, fontsize=12, y=1.02)
    fig.legend(handles, labels, loc='upper center',
               ncol=len(labels), bbox_to_anchor=(0.5, 1.0), fontsize=10)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    return fig

### Median of posterior-median θ across trajectories

For each trajectory and round, take the **median value of θ under the listener's posterior** (CDF crossing 0.5, with linear interpolation between grid points). Then take the median of those across trajectories.

In [ ]:
plot_belief_grid(
    summaries_post_med, TRUE_THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
    y_label=r'median $\theta^\star$ (posterior med)',
    fig_title='Median of posterior median θ across trajectories (95% CI shading)',
)
plt.show()


## 5. Compute — full posteriors + JS disagreement per round

For the heatmap and disagreement plots we need the **full distribution** at each cell, not a scalar summary.

- `belief_with_prior(belief_ds)` returns shape `(n_true_theta, n_traj, n_rounds, n_theta_grid)`. Round 0 is the uniform prior over the agents' grid.
- `mean_posterior` is the average distribution across trajectories at each `(true_theta, round, theta)`.
- `js_disagreement` is the average Jensen–Shannon divergence between every trajectory's posterior and the mean posterior:
  $$D_k = \frac{1}{n}\sum_{i=1}^n \mathrm{JS}\big(p_{i,k},\; \bar p_k\big).$$

JS is in **nats** (natural log). Divide by $\log 2$ to convert to bits if you prefer.


In [ ]:
def belief_with_prior(belief_ds):
    """Stack obs+utt into traj, prepend uniform prior at round 0.
    Returns shape (n_true_theta, n_traj, n_rounds, n_theta_grid)."""
    bt = belief_ds.belief_theta.stack(traj=('obs_idx', 'utt_idx')).transpose(
        'theta_true', 'traj', 't', 'theta'
    )
    arr = bt.values
    n_thetas, n_traj, _, n_grid = arr.shape
    prior_block = np.full((n_thetas, n_traj, 1, n_grid), 1.0 / n_grid)
    return np.concatenate([prior_block, arr], axis=2)


def _kl_safe(p, q, axis=-1):
    """KL(p || q) summed along axis; treats 0 * log(0/_) as 0 and drops infs."""
    with np.errstate(divide='ignore', invalid='ignore'):
        terms = p * (np.log(p) - np.log(q))
    return np.where(np.isfinite(terms), terms, 0.0).sum(axis=axis)


def js_disagreement(belief_arr):
    """Mean JS(p_traj, mean_p) per (true_theta, round). Returns (n_true_theta, n_rounds)."""
    mean_p = belief_arr.mean(axis=1, keepdims=True)
    m = 0.5 * (belief_arr + mean_p)
    js_per_traj = 0.5 * (_kl_safe(belief_arr, m) + _kl_safe(mean_p, m))
    return js_per_traj.mean(axis=1)


full_belief    = {k: belief_with_prior(ds)  for k, ds in beliefs.items()}
mean_posterior = {k: arr.mean(axis=1)       for k, arr in full_belief.items()}
js             = {k: js_disagreement(arr)   for k, arr in full_belief.items()}

for k in beliefs:
    print(
        f'{k}: mean_posterior {mean_posterior[k].shape} '
        f'in [{mean_posterior[k].min():.3f}, {mean_posterior[k].max():.3f}]; '
        f'js {js[k].shape} in [{js[k].min():.4f}, {js[k].max():.4f}]'
    )


## 6. Posterior heatmap — mean $\bar p_k(\theta_j)$ over rounds

Each panel: x = round, y = θ value (the agents' grid), color = mean $P(\theta \mid u_{0..t})$ across trajectories.

Diverging colormap (`RdBu_r`) centered at the prior probability $1/|\Theta| = 1/11 \approx 0.091$:

- **white** → mass at the prior level
- **red** → above-prior mass (concentration)
- **blue** → below-prior mass (mass moved away)

Black dashed line marks the true θ. Two figures, one per listener (each 2 rows × 9 cols), sharing a common colorbar scale so credulous and vigilant are directly comparable.


In [ ]:
from matplotlib.colors import TwoSlopeNorm


def plot_heatmap_grid(mean_post_per_speaker, theta_grid, true_thetas,
                      speakers, speaker_titles, vmax, listener_label):
    """
    mean_post_per_speaker : dict {spk_key: array (n_true_theta, n_rounds, n_theta_grid)}
    theta_grid            : 1-D array of agents' theta values (defines y-axis).
    """
    n_rows = len(speakers)
    n_cols = len(true_thetas)
    n_grid = len(theta_grid)
    vcenter = 1.0 / n_grid
    norm = TwoSlopeNorm(vcenter=vcenter, vmin=0.0, vmax=vmax)

    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(2.0 * n_cols, 2.5 * n_rows),
        sharex=True, sharey=True,
        squeeze=False,
    )
    n_rounds = next(iter(mean_post_per_speaker.values())).shape[1]

    im = None
    for row, spk_key in enumerate(speakers):
        mp = mean_post_per_speaker[spk_key]    # (n_true_theta, n_rounds, n_theta_grid)
        for col, theta_true in enumerate(true_thetas):
            ax = axes[row, col]
            arr = mp[col].T                    # (n_theta_grid, n_rounds): y=theta, x=round
            im = ax.imshow(
                arr,
                aspect='auto', origin='lower',
                extent=[-0.5, n_rounds - 0.5,
                        theta_grid[0] - 0.05, theta_grid[-1] + 0.05],
                norm=norm,
                cmap='RdBu_r',
            )
            ax.axhline(theta_true, color='black', linestyle='--',
                       linewidth=0.9, alpha=0.7)
            if row == 0:
                ax.set_title(f'$\\theta_{{true}}={theta_true}$', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'{speaker_titles[spk_key]}\nθ', fontsize=9)
            if row == n_rows - 1:
                ax.set_xlabel('round')

    if im is not None:
        cbar = fig.colorbar(im, ax=axes.ravel().tolist(),
                            shrink=0.85, pad=0.02, aspect=25)
        cbar.set_label(
            f'mean $P(\\theta | u_{{0..t}})$ — '
            f'white = prior {vcenter:.3f}'
        )
    fig.suptitle(f'{listener_label} — mean posterior heatmap', fontsize=12, y=1.0)
    return fig


# Use a global vmax across both heatmap figures so credulous and vigilant
# share the same color scale.
GLOBAL_VMAX = max(arr.max() for arr in mean_posterior.values())
THETA_GRID  = beliefs[('inf', 'credulous')].theta.values

print(f'GLOBAL_VMAX (mean P(θ)) = {GLOBAL_VMAX:.3f}')
print(f'Prior mass per θ = 1/{len(THETA_GRID)} = {1/len(THETA_GRID):.3f}')


In [ ]:
# Heatmap: credulous listener
credulous_means = {spk: mean_posterior[(spk, 'credulous')] for spk in SPEAKERS}
plot_heatmap_grid(
    credulous_means, THETA_GRID, TRUE_THETAS,
    speakers=list(SPEAKERS.keys()),
    speaker_titles=SPEAKER_TITLES,
    vmax=GLOBAL_VMAX,
    listener_label='Credulous listener',
)
plt.show()


In [ ]:
# Heatmap: vigilant listener
vigilant_means = {spk: mean_posterior[(spk, 'vigilant')] for spk in SPEAKERS}
plot_heatmap_grid(
    vigilant_means, THETA_GRID, TRUE_THETAS,
    speakers=list(SPEAKERS.keys()),
    speaker_titles=SPEAKER_TITLES,
    vmax=GLOBAL_VMAX,
    listener_label='Vigilant listener',
)
plt.show()


## 7. Disagreement curve — Jensen–Shannon divergence over rounds

The heatmap shows the mean posterior, which can hide multimodality (e.g., two clusters of trajectories converging to opposite corners would average to a flat shape). The disagreement curve $D_k$ catches that:

$$D_k = \frac{1}{n}\sum_{i=1}^n \mathrm{JS}\!\big(p_{i,k},\; \bar p_k\big).$$

- $D_0 = 0$: every trajectory shares the prior at round 0, no disagreement.
- $D_k > 0$ at later $k$: trajectories have moved to different posteriors. Larger $D_k$ means more spread / multimodality.

Two listeners overlaid per panel (blue = credulous, orange = vigilant), same 2 × 9 layout.


In [ ]:
def plot_js_grid(js_summaries, true_thetas, speakers, listeners,
                 listener_colors, speaker_titles):
    n_rows = len(speakers)
    n_cols = len(true_thetas)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(2.0 * n_cols, 2.5 * n_rows),
        sharex=True, sharey=True,
        squeeze=False,
    )
    any_arr = next(iter(js_summaries.values()))
    n_rounds = any_arr.shape[1]
    rounds = np.arange(n_rounds)

    for row, spk_key in enumerate(speakers):
        for col, theta_true in enumerate(true_thetas):
            ax = axes[row, col]
            for lst_key in listeners:
                arr = js_summaries[(spk_key, lst_key)]
                color = listener_colors[lst_key]
                ax.plot(rounds, arr[col], color=color, label=lst_key, linewidth=1.6)
            if row == 0:
                ax.set_title(f'$\\theta_{{true}}={theta_true}$', fontsize=10)
            if col == 0:
                ax.set_ylabel(f'{speaker_titles[spk_key]}\n$D_k$ (nats)', fontsize=9)
            if row == n_rows - 1:
                ax.set_xlabel('round')
            ax.set_xlim(0, n_rounds - 1)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=len(labels),
               bbox_to_anchor=(0.5, 1.0), fontsize=10)
    fig.suptitle(
        'Mean JS divergence between trajectory posteriors and mean posterior',
        fontsize=12, y=1.0,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    return fig


plot_js_grid(
    js, TRUE_THETAS,
    speakers=list(SPEAKERS.keys()),
    listeners=list(LISTENERS.keys()),
    listener_colors=LISTENER_COLORS,
    speaker_titles=SPEAKER_TITLES,
)
plt.show()


## What to look for

- **Truth line** (black dashed) marks $\theta_{true}$ for each column.
- Under the **informative speaker** (row 1), both listeners should track the truth — credulous is *correctly* assuming the speaker is informative.
- Under the **persuasive speaker** (row 2), credulous should be **biased**: it still assumes informativeness, so persuasive utterances move its summary up. The vigilant listener corrects for this.
- The posterior median is bounded by the θ grid's extent (0..1), unlike $\mathbb{E}[\theta]$ which is a weighted average. For unimodal posteriors near a single grid point, the posterior median ≈ MAP, and the cross-trajectory median is a robust central tendency that's relatively insensitive to outlier trajectories.